In [1]:
import pandas as pd
import numpy as np

batch_log=pd.read_csv("../data/raw/batch_log.csv", parse_dates=["StartTimestamp"])
process_parameters=pd.read_csv("../data/raw/process_parameters.csv")
quality_table=pd.read_csv("../data/raw/quality_table.csv")

### Schema Check

In [2]:
assert batch_log["Batch_ID"].is_unique
assert quality_table["Batch_ID"].is_unique
assert process_parameters["Batch_ID"].isin(batch_log["Batch_ID"]).all()

### Standardization (Strip and astype)

In [3]:
process_parameters['Step']=process_parameters['Step'].str.strip()
process_parameters['Param_Name']=process_parameters['Param_Name'].str.strip()
quality_table['OOS_Flag']=quality_table['OOS_Flag'].astype(int)

### Missing Value

In [4]:
assert process_parameters['Param_Value'].notnull().all()
assert quality_table['OOS_Flag'].notnull().all()

### Winsorization (Clipping Parameter_Value of each Parameter_Name)

In [5]:
specs_limits={
        'Mixing_Speed_RPM':(120,180), 
        'Mixing_Time_Min':(10,20), 
        'Blender_load_%':(60,75),
        'Granulation_Temp_C':(55,70), 
        'Granulation_Moisture_%':(2.0,4.0),
        'Granulation_Time_Min':(15,30), 
        'Coating_Spray_Rate':(5,10), 
        'Inlet_Temp_C':(45,60),
        'Coating_Time_Min':(20,40)
}

In [6]:
def clip_to_spec(row):
    low, high=specs_limits[row["Param_Name"]]
    return np.clip(row["Param_Value"],low, high)

process_parameters["Clipped_Param_Value"]=process_parameters.apply(clip_to_spec,axis=1)

### Pivot and Merge (Create a single Data Frame)

In [7]:
pivoted_params=process_parameters.pivot_table(index="Batch_ID", columns="Param_Name", values="Param_Value")
merged_all=batch_log.merge(pivoted_params, on="Batch_ID").merge(quality_table, on="Batch_ID")
merged_log_params=pivoted_params.merge(batch_log, on="Batch_ID")

In [8]:
pivoted_params

Param_Name,Blender_load_%,Coating_Spray_Rate,Coating_Time_Min,Granulation_Moisture_%,Granulation_Temp_C,Granulation_Time_Min,Inlet_Temp_C,Mixing_Speed_RPM,Mixing_Time_Min
Batch_ID,,,,,,,,,
BATCH_0001,64.1951,7.7535,32.0761,2.4827,62.5640,20.5537,59.3939,124.9493,12.5048
BATCH_0002,70.0006,7.6885,32.9209,2.8915,63.7867,18.9745,56.7596,122.4041,13.8348
BATCH_0003,56.6179,10.4212,38.8662,3.8597,68.1738,22.3567,58.4942,175.9589,17.2952
BATCH_0004,60.5145,8.0295,31.1342,2.7301,57.3226,24.3476,53.6096,160.6256,11.3101
BATCH_0005,69.7215,8.4464,36.7114,2.6197,70.1598,14.1825,48.8420,140.1366,14.3277
...,...,...,...,...,...,...,...,...,...
BATCH_2996,64.2908,6.9663,29.3519,2.1360,61.2951,29.0460,46.3982,144.6094,16.0971
BATCH_2997,65.1828,9.5451,24.9349,2.6168,59.8313,20.7728,48.5186,136.2273,13.4538
BATCH_2998,67.1262,7.2713,26.0878,2.8179,60.3138,13.5630,46.6020,158.9730,14.0584


In [9]:
merged_all

,Batch_ID,Product_ID,Line_ID,Operator,StartTimestamp,Blender_load_%,Coating_Spray_Rate,Coating_Time_Min,Granulation_Moisture_%,Granulation_Temp_C,Granulation_Time_Min,Inlet_Temp_C,Mixing_Speed_RPM,Mixing_Time_Min,Assay,Dissolution,Impurity,Hardness,OOS_Flag
0,BATCH_0001,CardioStat_IR,Line_2,Operator_A,2023-01-01 06:00:00,64.1951,7.7535,32.0761,2.4827,62.5640,20.5537,59.3939,124.9493,12.5048,96.145266,90.0,0.200000,7,1
1,BATCH_0002,CardioStat_IR,Line_2,Operator_B,2023-01-01 16:00:00,70.0006,7.6885,32.9209,2.8915,63.7867,18.9745,56.7596,122.4041,13.8348,95.873683,90.0,0.200000,7,1
2,BATCH_0003,CardioStat_IR,Line_2,Operator_B,2023-01-02 00:00:00,56.6179,10.4212,38.8662,3.8597,68.1738,22.3567,58.4942,175.9589,17.2952,100.000000,90.0,0.200000,7,0
3,BATCH_0004,NeuroCalm_IR,Line_2,Operator_D,2023-01-02 10:00:00,60.5145,8.0295,31.1342,2.7301,57.3226,24.3476,53.6096,160.6256,11.3101,100.000000,90.0,0.200000,7,0
4,BATCH_0005,NeuroCalm_IR,Line_1,Operator_A,2023-01-02 20:00:00,69.7215,8.4464,36.7114,2.6197,70.1598,14.1825,48.8420,140.1366,14.3277,100.000000,90.0,0.508155,7,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,BATCH_2996,GastroEase_CR,Line_2,Operator_D,2026-01-25 03:00:00,64.2908,6.9663,29.3519,2.1360,61.2951,29.0460,46.3982,144.6094,16.0971,100.000000,90.0,0.200000,7,0
2996,BATCH_2997,GastroEase_CR,Line_1,Operator_A,2026-01-25 13:00:00,65.1828,9.5451,24.9349,2.6168,59.8313,20.7728,48.5186,136.2273,13.4538,100.000000,90.0,0.200000,7,0
2997,BATCH_2998,CardioStat_IR,Line_1,Operator_C,2026-01-25 21:00:00,67.1262,7.2713,26.0878,2.8179,60.3138,13.5630,46.6020,158.9730,14.0584,100.000000,90.0,0.200000,7,0
2998,BATCH_2999,NeuroCalm_IR,Line_2,Operator_C,2026-01-26 05:00:00,89.7713,5.9213,36.8964,2.4622,57.5501,16.7844,53.6137,131.6010,11.9332,100.000000,90.0,0.200000,7,0


In [10]:
merged_log_params

,Batch_ID,Blender_load_%,Coating_Spray_Rate,Coating_Time_Min,Granulation_Moisture_%,Granulation_Temp_C,Granulation_Time_Min,Inlet_Temp_C,Mixing_Speed_RPM,Mixing_Time_Min,Product_ID,Line_ID,Operator,StartTimestamp
0,BATCH_0001,64.1951,7.7535,32.0761,2.4827,62.5640,20.5537,59.3939,124.9493,12.5048,CardioStat_IR,Line_2,Operator_A,2023-01-01 06:00:00
1,BATCH_0002,70.0006,7.6885,32.9209,2.8915,63.7867,18.9745,56.7596,122.4041,13.8348,CardioStat_IR,Line_2,Operator_B,2023-01-01 16:00:00
2,BATCH_0003,56.6179,10.4212,38.8662,3.8597,68.1738,22.3567,58.4942,175.9589,17.2952,CardioStat_IR,Line_2,Operator_B,2023-01-02 00:00:00
3,BATCH_0004,60.5145,8.0295,31.1342,2.7301,57.3226,24.3476,53.6096,160.6256,11.3101,NeuroCalm_IR,Line_2,Operator_D,2023-01-02 10:00:00
4,BATCH_0005,69.7215,8.4464,36.7114,2.6197,70.1598,14.1825,48.8420,140.1366,14.3277,NeuroCalm_IR,Line_1,Operator_A,2023-01-02 20:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,BATCH_2996,64.2908,6.9663,29.3519,2.1360,61.2951,29.0460,46.3982,144.6094,16.0971,GastroEase_CR,Line_2,Operator_D,2026-01-25 03:00:00
2996,BATCH_2997,65.1828,9.5451,24.9349,2.6168,59.8313,20.7728,48.5186,136.2273,13.4538,GastroEase_CR,Line_1,Operator_A,2026-01-25 13:00:00
2997,BATCH_2998,67.1262,7.2713,26.0878,2.8179,60.3138,13.5630,46.6020,158.9730,14.0584,CardioStat_IR,Line_1,Operator_C,2026-01-25 21:00:00
2998,BATCH_2999,89.7713,5.9213,36.8964,2.4622,57.5501,16.7844,53.6137,131.6010,11.9332,NeuroCalm_IR,Line_2,Operator_C,2026-01-26 05:00:00


In [11]:
print(f"Overall average OOS_Flags: {merged_all['OOS_Flag'].mean()}")
print(f"Shape of the final Data Frame is: {merged_all.shape}")

Overall average OOS_Flags: 0.21533333333333332
Shape of the final Data Frame is: (3000, 19)


### Saving as CSV

In [12]:
pivoted_params.to_csv("../data/cleaned/pivoted_process_parameter_table.csv", index=False)
merged_all.to_csv("../data/cleaned/merged_all.csv", index=False)
merged_log_params.to_csv("../data/cleaned/merged_log_params.csv", index=False)
